### <b>2강 pandas로 데이터 다루기 - read_csv · 열 선택 · 필터 · groupby · 결측치</b>

* 실습 데이터 두 개(<b>students.csv</b>, <b>cafe_sales.csv</b>)를 읽고, 표의 모양과 자료형을 확인한다.
* 열은 이름으로, 행은 조건으로 고르고, 범주별로 묶어 개수·평균을 낸다.
* 그리기 전에 빈 값과 튀는 값을 찾아 두고, 0처럼 이상해 보이는 값이 실제 값인지 판단한다.

* 첫 코드 셀은 매 시간 실행하는 <b>설정 셀</b>이다. 1강에서 정한 그대로다.
  * 오늘은 그래프를 그리지 않지만, 설정 셀은 매 시간 똑같이 먼저 실행한다.

In [ ]:
import platform
import pandas as pd
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

#### <b>1) 두 CSV 읽고 모양과 자료형 보기</b>

* 이 과정의 실습 데이터는 두 파일이다. 둘 다 이 과정을 위해 만든 <b>가상 데이터</b>로, 실제 학교·카페·학생과 관계가 없다.
  * <b>students.csv</b>: 대학생 300명의 생활 습관 설문. 학생 1명이 1행이다.
  * <b>cafe_sales.csv</b>: 캠퍼스 카페의 1년(2025-03-01 ~ 2026-02-28) 일별 판매. 하루가 1행이다.
* <b>CSV</b>: 쉼표(,)로 칸을 나눈 표 파일이다. 메모장으로 열면 첫 줄에 열 이름, 그 아래 줄마다 값이 있다.
* <b>pd.read_csv('경로')</b>: CSV를 읽어 <b>DataFrame</b>(행과 열로 된 표)으로 만든다.
  * <b>'../data/students.csv'</b>: 노트북 폴더의 한 칸 위(<b>..</b>)에 있는 data 폴더 안의 파일이다.
* <b>head()</b>: 표의 앞 5행을 보여 준다. 왼쪽 굵은 숫자 0, 1, 2…는 행 번호(<b>index</b>)다.

* <b>shape</b>: 표의 모양을 <b>(행 수, 열 수)</b>로 알려 준다. 괄호 없이 쓴다.
  * students는 300행 12열이다. 학생 300명, 항목 12개라고 읽는다.

* <b>info()</b>: 열마다 이름, 빈 값이 아닌 개수(<b>Non-Null Count</b>), 자료형(<b>Dtype</b>)을 한 번에 보여 준다.
* <b>자료형(dtype)</b>: 열에 든 값의 종류다. 그래프를 그릴 수 있는 열인지 여기서 먼저 확인한다.
  * <b>int64</b>: 정수(학년, 점수), <b>float64</b>: 소수(공부 시간, 수면 시간)
  * <b>str</b>: 글자(전공, 주거 형태). pandas 3부터 글자 열은 <b>str</b>로 표시된다. 예전 자료에서는 <b>object</b>로 나온다.
* sleep_hours는 285, sns_minutes는 294로 300보다 적다. 빈 값이 있다는 뜻이다(6)에서 다시 본다).

* cafe_sales.csv의 <b>date</b> 열은 날짜다. 그런데 그냥 읽으면 pandas는 이 열을 글자(<b>str</b>)로 읽는다.
  * 글자로 된 날짜는 '2025-03-01'이라는 문자열일 뿐이라, 날짜 순서나 간격을 모른다.

* <b>parse_dates=['date']</b>: 읽을 때 date 열을 날짜 자료형(<b>datetime64</b>)으로 바꾼다.
  * 날짜 자료형이어야 3강 선 그래프에서 x축이 날짜 순서와 간격대로 그려진다.
* <b>dtypes</b>: 열별 자료형만 따로 보여 준다. cafe는 365행 14열이다.
  * 맨 아래 <b>dtype: object</b>는 dtypes 결과 자체의 자료형이라 열 자료형과 관계없다.

* 앞 5행을 보면 하루가 한 줄이고, 손님 수(customers), 메뉴별 잔 수(americano·latte·ade·tea), 매출(sales)이 열로 있다.
  * year_month는 '2025-03' 같은 연-월 글자, period는 학사 기간(1학기·여름방학·2학기·겨울방학), exam_week는 시험 주간이면 1이다.

#### <b>2) 시험 주간(exam_week == 1)만 골라 손님 수 보기</b>

* <b>df['열 이름']</b>: 열 하나를 고른다. 결과는 <b>Series</b>(이름표가 붙은 값 한 줄)다.
  * 열 이름은 철자·대소문자까지 같아야 한다. 틀리면 <b>KeyError</b>가 난다.

* <b>df[['열1', '열2']]</b>: 여러 열을 고른다. 대괄호 안에 <b>열 이름 리스트</b>를 넣으므로 대괄호가 두 겹이다.
  * 결과는 다시 DataFrame(표)이다.

* <b>조건식</b>: <b>cafe['exam_week'] == 1</b>처럼 열과 값을 비교하면 행마다 <b>True/False</b>가 담긴 Series가 나온다.
  * <b>sum()</b>을 하면 True를 1로 세므로, 조건에 맞는 행의 개수가 된다. 시험 주간은 28일이다.

* <b>df[조건]</b>: 조건이 True인 행만 남긴 새 표를 만든다. 이것을 <b>필터</b>라고 한다.
  * 결과 표를 변수에 담아 두면 이어서 열을 고르거나 평균을 낼 수 있다.

* 시험 주간과 나머지 날의 평균 손님 수를 비교한다. <b>mean()</b>은 평균, <b>round(값, 2)</b>는 소수 2자리 반올림이다.
  * 시험 주간 평균은 455.07명, 나머지 날은 252.4명이다.

* <b>df.loc[행 조건, 열 이름]</b>: 행과 열을 한 번에 고른다. 쉼표 앞이 행, 뒤가 열이다.
  * 위의 <b>exam[['date', 'weekday', 'customers']]</b>를 한 줄로 쓴 것과 같다.
  * 값을 <b>바꿀 때</b>도 <b>.loc</b>을 쓴다. 6)에서 직접 확인한다.

#### <b>3) 공학 전공이면서 기숙사에 사는 학생만 고르기</b>

* 조건 두 개를 묶을 때는 <b>&</b>(그리고), <b>|</b>(또는)를 쓴다.
  * 조건 하나하나를 <b>괄호</b>로 감싼다. 괄호를 빼면 TypeError가 난다.
  * 파이썬의 <b>and</b>·<b>or</b>는 쓰지 않는다. 쓰면 ValueError(The truth value of a Series is ambiguous)가 난다.
* 공학이면서 기숙사인 학생은 22명이다.

* <b>|</b>(또는): 자취이거나 기숙사인 학생을 고른다. 둘 중 하나만 맞아도 남는다.
  * 자취 115명 + 기숙사 85명 = 200명이다.

* <b>isin(리스트)</b>: 값이 리스트 안에 있으면 True다. 같은 열에 '또는'을 여러 번 쓸 때 짧게 쓴다.
  * 위의 | 조건과 결과(200명)가 같다.

* 고른 표에서도 열을 골라 평균을 낼 수 있다. 공학·기숙사 학생의 평균 공부 시간은 18.2시간으로, 전체 평균 12.61시간보다 길다.

#### <b>4) 전공별 학생 수와 평균 공부 시간</b>

* <b>value_counts()</b>: 열의 값마다 몇 번 나오는지 세어, 많은 순으로 보여 준다.
  * 공학 80, 사회과학 70, 자연과학 55, 인문 55, 예체능 40명이다.
  * 자연과학과 인문은 55명으로 같다. 같은 개수끼리의 순서는 바뀔 수 있다.

* <b>groupby('묶을 열')['계산할 열'].mean()</b>: 묶을 열의 값이 같은 행끼리 모아, 계산할 열의 평균을 낸다.
  * 읽는 순서: 전공별로 묶어서 → 공부 시간 열을 골라 → 평균
  * 계산할 열을 고르지 않고 <b>students.groupby('major').mean()</b>을 하면 TypeError가 난다. 글자 열(student_id·housing)은 평균을 낼 수 없기 때문이다.

* 결과는 전공 이름의 가나다순이다. 비교하기 쉽게 정렬한다.
  * <b>round(2)</b>: 소수 2자리 반올림, <b>sort_values(ascending=False)</b>: 큰 값부터
* 공학 18.16시간이 가장 길고 예체능 6.28시간이 가장 짧다. 이 표가 4강 막대 그래프의 재료가 된다.

* <b>agg(['mean', 'count'])</b>: 여러 계산을 한 번에 한다. 결과는 계산마다 열이 하나씩 생긴 표다.
  * <b>sort_values('mean', ascending=False)</b>: 표는 정렬할 열 이름을 넣는다.
  * count 열은 value_counts()의 학생 수와 같다.

#### <b>5) 기간(period)별·요일별 평균 손님 수</b>

* 같은 방법으로 cafe를 학사 기간별로 묶어 평균 손님 수를 낸다.
  * 결과가 1학기 → 2학기 → 겨울방학 → 여름방학 순이다. 시간 순서가 아니라 <b>가나다순</b>이다.

* <b>reindex(순서 리스트)</b>: 결과의 행을 리스트에 적은 순서로 다시 놓는다.
  * 시간 순서(1학기 → 여름방학 → 2학기 → 겨울방학)로 두어야 그래프로 옮길 때 흐름이 바르게 보인다.
* 1학기 364.00명, 2학기 332.91명으로 학기 중이 높고, 여름방학 149.11명, 겨울방학 136.30명으로 방학이 낮다.

* 요일도 가나다순(금·목·수·월…)으로 나오므로 월~일 순서로 다시 놓는다.
* 수요일 374.10명이 가장 많고, 일요일 113.79명이 가장 적다. 주말에 크게 떨어진다.
  * 이 평균에는 카페가 쉰 날의 0이 섞여 있다. 6)에서 확인한다.

#### <b>6) 빈 값 세기와 튀는 값 찾기</b>

* <b>결측치(NaN)</b>: 비어 있는 값이다. 설문에 답하지 않은 칸처럼 값이 없는 자리다.
* <b>isna()</b>: 칸마다 비어 있으면 True다. <b>sum()</b>을 붙이면 열별로 빈 칸 수를 센다.
  * sleep_hours 15개, sns_minutes 6개가 비어 있고, 나머지 열은 0개다.

* <b>dropna(subset=['열'])</b>: 그 열이 비어 있는 행을 뺀 <b>새 표</b>를 돌려준다. 원래 students는 그대로다.
  * 수면 시간을 그릴 때는 sleep_hours가 빈 15행을 빼고 285행으로 그린다.

* <b>이상치</b>: 다른 값과 동떨어진 값이다. <b>describe()</b>로 개수·평균·최솟값·사분위수·최댓값을 보면 찾기 쉽다.
  * <b>25% · 50% · 75%</b>: 값을 작은 순으로 줄 세웠을 때 4분의 1, 가운데, 4분의 3 자리의 값이다.
  * sns_minutes는 75%가 180분인데 최댓값(max)이 900분이다. commute_minutes는 75%가 50분인데 최댓값이 180분이다.

* 최댓값이 누구인지 필터로 찾는다. SNS를 하루 500분 넘게 쓴 학생은 <b>S026</b> 한 명(900분)이다.
  * 900분은 하루 15시간이다. 그다음으로 큰 값은 480분이다.

* <b>sort_values('열', ascending=False).head(3)</b>: 큰 값부터 3행을 본다.
  * 통학 시간 1위는 <b>S037</b>의 180분이고, 2위는 114분으로 차이가 크다.
* 이상치는 바로 지우지 않는다. 실제로 그런 학생일 수도, 입력 실수일 수도 있으므로 찾아서 적어 둔다.
  * 6강 박스플롯에서 이 값들이 따로 떨어진 점으로 다시 보인다.

* cafe에는 빈 값이 없다. 대신 손님 수가 <b>0</b>인 날이 있다.
  * 7일 모두 판매 값이 0이고, 날짜가 2025년 추석 연휴(10/5~10/8)와 2026년 설 연휴(2/16~2/18)다.
* 0은 잘못 들어간 값이 아니라 <b>카페가 쉰 날의 실제 값</b>이다. 지우지 않고 휴무일이라고 표시해 둔다.

* 먼저 모든 날을 '영업'으로 적은 status 열을 만들고, 손님이 0인 날만 '휴무'로 바꿔 본다.
* 아래처럼 <b>필터한 표에 다시 열을 골라 값을 넣는 방식</b>(연쇄 할당)은 원래 cafe를 바꾸지 못한다.
  * pandas 3은 <b>Copy-on-Write</b>(고른 결과는 원본과 따로인 복사본처럼 동작) 방식이라, 값이 복사본에만 들어간다.
  * <b>ChainedAssignmentError</b> 경고가 나오고, '휴무'가 0개 그대로다. 이 경고는 일부러 보여 주는 것이다.

* <b>df.loc[행 조건, '열'] = 값</b>: 행과 열을 한 번에 지정해 원래 표의 값을 바꾼다. 값을 바꿀 때는 이 방식만 쓴다.
  * 이제 휴무 7일, 영업 358일이다.

* 영업한 날만 골라 요일별 평균 손님 수를 다시 낸다.
  * 휴무일이 있던 월·화·수·일요일의 평균이 올라간다. 수요일은 374.10 → 389.06명이다.
  * 쉰 날을 넣을지 뺄지는 질문에 따라 정한다. 표시해 두면 둘 다 바로 계산할 수 있다.
* 오늘 만든 표(value_counts·groupby 결과)를 3강부터 그래프로 옮긴다.